# ✅ STEP-BY-STEP CUSTOMER CHURN PREDICTION PROJECT

This notebook predicts whether customers will leave a business using machine learning.

In [ ]:
# 🔹 Step 1: Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# 🔹 Step 2: Load and Explore Data
# Load the BankChurners dataset which contains customer information and churn status

df = pd.read_csv("BankChurners.csv")

print("📊 DATASET INFORMATION")
print("=" * 80)
print(f"Total records: {df.shape[0]}")
print(f"Total features: {df.shape[1]}")
print("\n📋 First few rows:")
print(df.head())
print("\n📈 Dataset Info:")
print(df.info())


Dataset Shape: (10127, 23)

First few rows:
   CLIENTNUM     Attrition_Flag  Customer_Age Gender  Dependent_count  \
0  768805383  Existing Customer            45      M                3   
1  818770008  Existing Customer            49      F                5   
2  713982108  Existing Customer            51      M                3   
3  769911858  Existing Customer            40      F                4   
4  709106358  Existing Customer            40      M                3   

  Education_Level Marital_Status Income_Category Card_Category  \
0     High School        Married     $60K - $80K          Blue   
1        Graduate         Single  Less than $40K          Blue   
2        Graduate        Married    $80K - $120K          Blue   
3     High School        Unknown  Less than $40K          Blue   
4      Uneducated        Married     $60K - $80K          Blue   

   Months_on_book  ...  Credit_Limit  Total_Revolving_Bal  Avg_Open_To_Buy  \
0              39  ...       12691.0      

In [ ]:
# 🔹 Step 3: Data Cleaning and Preprocessing
# This step prepares raw data for machine learning models

print("\n🧹 DATA CLEANING")
print("=" * 80)

# 3.1: Remove Unnecessary Columns (ID columns don't help predictions)
df = df.drop("CLIENTNUM", axis=1)
print("✓ Removed CLIENTNUM column")

# 3.2: Convert Target Variable to Binary (1 = Churn, 0 = Stay)
df["Attrition_Flag"] = df["Attrition_Flag"].map({
    "Attrited Customer": 1,  # Customer left
    "Existing Customer": 0   # Customer stayed
})
print("✓ Converted Attrition_Flag to binary (0/1)")

# 3.3: Handle Missing Values
initial_rows = len(df)
df = df.dropna()
removed_rows = initial_rows - len(df)
print(f"✓ Removed {removed_rows} rows with missing values")

# 3.4: Convert Categorical Data to Numeric (One-Hot Encoding)
# This converts text categories into binary columns (e.g., Gender_M, Gender_F)
df = pd.get_dummies(df, drop_first=True)
print("✓ One-hot encoded categorical variables")

print(f"\n✅ Data Cleaning Complete!")
print(f"Final dataset shape: {df.shape}")
print(f"Churn Distribution: {df['Attrition_Flag'].value_counts().to_dict()}")


✓ Data Cleaning Complete
Cleaned Data Shape: (10127, 35)


In [ ]:
# 🔹 Step 4: Define Features (X) and Target (y)
# Features (X): All columns except the target variable
# Target (y): What we want to predict (Attrition_Flag)

X = df.drop("Attrition_Flag", axis=1)  # All features
y = df["Attrition_Flag"]                # Target variable

print("\n🎯 FEATURES AND TARGET")
print("=" * 80)
print(f"Number of features (X): {X.shape[1]}")
print(f"Number of samples (y): {y.shape[0]}")
print(f"\nTarget variable distribution:")
print(y.value_counts())


Features shape: (10127, 34)
Target shape: (10127,)


In [ ]:
# 📊 Visualize Churn Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Churn Distribution (Pie Chart)
churn_counts = df['Attrition_Flag'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # Green for Stay, Red for Churn
axes[0].pie(churn_counts, labels=['Existing Customer', 'Attrited Customer'], 
            autopct='%1.1f%%', colors=colors, startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})
axes[0].set_title('Customer Churn Distribution', fontsize=14, weight='bold')

# Plot 2: Churn Count (Bar Chart)
churn_counts.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Churn Count by Category', fontsize=14, weight='bold')
axes[1].set_ylabel('Number of Customers', fontsize=12)
axes[1].set_xlabel('Customer Status', fontsize=12)
axes[1].set_xticklabels(['Existing (0)', 'Attrited (1)'], rotation=0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📈 Churn Rate: {(churn_counts[1] / len(df) * 100):.2f}%")


In [ ]:
# 🔹 Step 5: Split Data into Training and Testing Sets
# 80% for training (model learns patterns)
# 20% for testing (evaluate model performance)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,        # 20% test data
    random_state=42,      # Reproducible results
    stratify=y            # Maintain class distribution
)

print("\n📋 DATA SPLIT")
print("=" * 80)
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing samples:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"Total samples:    {len(X)}")


Training samples: 8101
Testing samples: 2026


In [ ]:
# 🔹 Step 6: Feature Scaling (Normalization)
# Logistic Regression performs better with scaled features
# Scaling: Transform all features to similar ranges (0-1 or -1 to 1)
# Note: We fit scaler on TRAINING data only, then apply to test data

scaler = StandardScaler()

# Fit on training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler (don't fit again!)
X_test_scaled = scaler.transform(X_test)

print("\n📏 FEATURE SCALING")
print("=" * 80)
print("✓ StandardScaler fitted on training data")
print("✓ Test data scaled using training statistics")
print(f"Training set shape after scaling: {X_train_scaled.shape}")
print(f"Test set shape after scaling: {X_test_scaled.shape}")


✓ Features scaled successfully


In [ ]:
# 🔹 Step 7: Train Logistic Regression Model
# Logistic Regression: Fast, interpretable linear classification model
# class_weight="balanced": Handles imbalanced classes better

print("\n🤖 TRAINING LOGISTIC REGRESSION")
print("=" * 80)

log_model = LogisticRegression(
    max_iter=1000,           # Maximum iterations to converge
    class_weight="balanced"  # Handle imbalanced classes
)

# Train the model on scaled training data
log_model.fit(X_train_scaled, y_train)

# Make predictions on test data
y_pred_log = log_model.predict(X_test_scaled)

# Calculate accuracy
acc_log = accuracy_score(y_test, y_pred_log)

print(f"\n✅ Logistic Regression Model Trained!")
print(f"Accuracy: {acc_log:.4f} ({acc_log*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log, 
                          target_names=['No Churn (0)', 'Churn (1)']))



Logistic Regression Results
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1701
           1       1.00      1.00      1.00       325

    accuracy                           1.00      2026
   macro avg       1.00      1.00      1.00      2026
weighted avg       1.00      1.00      1.00      2026



In [ ]:
# 🔹 Step 8: Train Random Forest Model
# Random Forest: Ensemble of 300 decision trees
# Captures complex, non-linear patterns in data
# More powerful but slower than Logistic Regression

print("\n🤖 TRAINING RANDOM FOREST")
print("=" * 80)

rf_model = RandomForestClassifier(
    n_estimators=300,   # Number of trees in the forest
    random_state=42     # Reproducible results
)

# Train the model (Note: NO scaling needed for tree-based models)
rf_model.fit(X_train, y_train)

# Make predictions on test data
y_pred_rf = rf_model.predict(X_test)

# Calculate accuracy
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"\n✅ Random Forest Model Trained!")
print(f"Accuracy: {acc_rf:.4f} ({acc_rf*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, 
                          target_names=['No Churn (0)', 'Churn (1)']))



Random Forest Results
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1701
           1       1.00      1.00      1.00       325

    accuracy                           1.00      2026
   macro avg       1.00      1.00      1.00      2026
weighted avg       1.00      1.00      1.00      2026



In [ ]:
# 🔹 Step 9: Model Comparison and Selection
# Compare both models and select the best one based on accuracy

print("\n📊 MODEL COMPARISON")
print("=" * 80)

# Select the model with higher accuracy
if acc_rf > acc_log:
    final_model = rf_model
    model_name = "Random Forest"
    print(f"🏆 WINNER: Random Forest!")
else:
    final_model = log_model
    model_name = "Logistic Regression"
    print(f"🏆 WINNER: Logistic Regression!")

print(f"\nLogistic Regression Accuracy:  {acc_log:.4f} ({acc_log*100:.2f}%)")
print(f"Random Forest Accuracy:        {acc_rf:.4f} ({acc_rf*100:.2f}%)")
print(f"Difference:                    {abs(acc_rf - acc_log):.4f}")

# Visualize model comparison
fig, ax = plt.subplots(figsize=(10, 6))
models = ['Logistic Regression', 'Random Forest']
accuracies = [acc_log, acc_rf]
colors_bar = ['#3498db', '#e74c3c']

bars = ax.bar(models, accuracies, color=colors_bar, alpha=0.8, edgecolor='black', linewidth=2)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.4f}\n({acc*100:.2f}%)',
            ha='center', va='bottom', fontsize=12, weight='bold')

ax.set_ylim([0, 1.1])
ax.set_ylabel('Accuracy Score', fontsize=12, weight='bold')
ax.set_title('Model Accuracy Comparison', fontsize=14, weight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()



Best Model: Logistic Regression
Logistic Regression Accuracy: 1.0
Random Forest Accuracy: 1.0


In [ ]:
# 🔹 Step 10: Feature Importance Analysis
# Show which features are most important for predicting churn
# Random Forest can calculate the importance of each feature

print("\n📈 FEATURE IMPORTANCE")
print("=" * 80)

importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(10)

print("Top 10 Most Important Features:")
for i, (feature, importance) in enumerate(top_features.items(), 1):
    print(f"{i:2d}. {feature:<30s} : {importance:.4f}")

# Visualize feature importance
fig, ax = plt.subplots(figsize=(12, 6))
top_features.plot(kind='barh', ax=ax, color='#3498db', edgecolor='black', linewidth=1.5)
ax.set_xlabel('Importance Score', fontsize=12, weight='bold')
ax.set_ylabel('Features', fontsize=12, weight='bold')
ax.set_title('Top 10 Most Important Features for Churn Prediction', fontsize=14, weight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(top_features.values):
    ax.text(v, i, f' {v:.4f}', va='center', fontsize=10, weight='bold')

plt.tight_layout()
plt.show()



Top 5 Important Features:
Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2    0.412186
Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1    0.371343
Total_Trans_Ct                                                                                                                        0.044993
Total_Trans_Amt                                                                                                                       0.036931
Total_Revolving_Bal                                                                                                                   0.034193
dtype: float64


In [ ]:
# 📊 Visualize Confusion Matrices
# Confusion Matrix shows True Positives, False Positives, True Negatives, False Negatives

print("\n🎯 CONFUSION MATRICES")
print("=" * 80)

cm_log = confusion_matrix(y_test, y_pred_log)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression Confusion Matrix
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            cbar_kws={'label': 'Count'}, linewidths=2, linecolor='black')
axes[0].set_title('Logistic Regression - Confusion Matrix', fontsize=12, weight='bold')
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_xticklabels(['No Churn', 'Churn'])
axes[0].set_yticklabels(['No Churn', 'Churn'])

# Random Forest Confusion Matrix
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1], 
            cbar_kws={'label': 'Count'}, linewidths=2, linecolor='black')
axes[1].set_title('Random Forest - Confusion Matrix', fontsize=12, weight='bold')
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_xticklabels(['No Churn', 'Churn'])
axes[1].set_yticklabels(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

print("\n📋 Confusion Matrix Interpretation:")
print("   True Negatives (TN):  Correctly predicted 'No Churn'")
print("   False Positives (FP): Wrongly predicted 'Churn' (should be 'No Churn')")
print("   False Negatives (FN): Wrongly predicted 'No Churn' (should be 'Churn')")
print("   True Positives (TP):  Correctly predicted 'Churn'")


In [ ]:
# 🔹 Step 13: Test Predictions on Actual Dataset Examples
# Select one example where model predicts "Stay" and one where it predicts "Leave"

print("\n" + "="*80)
print("REAL CUSTOMER PREDICTIONS FROM TEST SET")
print("="*80)

# Helper function to predict a single customer
def predict_customer(row):
    """Make prediction for a single customer"""
    sample_row = row.to_frame().T
    if model_name == "Logistic Regression":
        sample_row = scaler.transform(sample_row)
    prediction_value = final_model.predict(sample_row)[0]
    return prediction_value

# Find examples from test set
stay_idx = None
leave_idx = None

for idx in X_test.index:
    pred_value = predict_customer(X_test.loc[idx])
    if pred_value == 0 and stay_idx is None:
        stay_idx = idx
    elif pred_value == 1 and leave_idx is None:
        leave_idx = idx
    if stay_idx is not None and leave_idx is not None:
        break

print("\n✅ Example 1: Customer Predicted to STAY")
print("-" * 80)
if stay_idx is not None:
    actual_value = y_test.loc[stay_idx]
    prediction_result = "✓ CORRECT" if actual_value == 0 else "✗ WRONG"
    print(f"Prediction: Customer will Stay")
    print(f"Actual Status: {'Customer Stayed' if actual_value == 0 else 'Customer Left'}")
    print(f"Result: {prediction_result}")
else:
    print("No stay example found in the test set.")

print("\n❌ Example 2: Customer Predicted to LEAVE")
print("-" * 80)
if leave_idx is not None:
    actual_value = y_test.loc[leave_idx]
    prediction_result = "✓ CORRECT" if actual_value == 1 else "✗ WRONG"
    print(f"Prediction: Customer will Leave")
    print(f"Actual Status: {'Customer Left' if actual_value == 1 else 'Customer Stayed'}")
    print(f"Result: {prediction_result}")
else:
    print("No leave example found in the test set.")


TWO EXAMPLE CUSTOMER PREDICTIONS

Example 1: Customer likely to STAY
--------------------------------------------------------------------------------
Row index: 2919
Model prediction: Customer will Stay
Actual status: Customer Stayed

Example 2: Customer likely to LEAVE
--------------------------------------------------------------------------------
Row index: 8623
Model prediction: Customer will Leave
Actual status: Customer Left


In [ ]:
# 🔹 Step 14: Make Predictions on New Customers
# Demonstrate how to make predictions on new customer data with both models

print("\n" + "="*80)
print("NEW CUSTOMER PREDICTIONS WITH PROBABILITIES")
print("="*80)

# Set up model references
log_reg = log_model
random_forest = rf_model
final_model_name = model_name
target_col = "Churn" if "Churn" in df.columns else "Attrition_Flag"
feature_columns = X.columns

def prepare_customer(raw_customer):
    """
    Convert raw customer data into model-ready format
    Handles: cleaning, encoding, and alignment with training data
    """
    row_df = pd.DataFrame([raw_customer]).copy()

    # Remove ID columns
    if "clientnum" in row_df.columns:
        row_df = row_df.drop("clientnum", axis=1)
    if "CLIENTNUM" in row_df.columns:
        row_df = row_df.drop("CLIENTNUM", axis=1)
    if "Attrition_Flag" in row_df.columns:
        row_df = row_df.drop("Attrition_Flag", axis=1)

    # Handle numeric conversions
    if "TotalCharges" in row_df.columns:
        row_df["TotalCharges"] = pd.to_numeric(row_df["TotalCharges"], errors="coerce")

    # Feature engineering
    if "tenure" in row_df.columns and "TotalCharges" in row_df.columns:
        row_df["AvgMonthlySpend"] = (row_df["TotalCharges"] / row_df["tenure"].replace(0, 1)).round(2)
    if "tenure" in row_df.columns:
        row_df["IsNewCustomer"] = (row_df["tenure"] <= 12).astype(int)
    if "PhoneService" in row_df.columns:
        row_df["ServiceCount"] = (
            (row_df["PhoneService"] == "Yes").astype(int)
            + (row_df.get("InternetService", "No") != "No").astype(int)
            + (row_df.get("OnlineSecurity", "No") == "Yes").astype(int)
            + (row_df.get("OnlineBackup", "No") == "Yes").astype(int)
            + (row_df.get("DeviceProtection", "No") == "Yes").astype(int)
            + (row_df.get("TechSupport", "No") == "Yes").astype(int)
            + (row_df.get("StreamingTV", "No") == "Yes").astype(int)
            + (row_df.get("StreamingMovies", "No") == "Yes").astype(int)
        )

    # One-hot encode categorical variables
    row_df = pd.get_dummies(row_df, drop_first=True)
    
    # Align with training data columns (add missing, drop extra)
    row_df = row_df.reindex(columns=feature_columns, fill_value=0)
    return row_df

def show_prediction(title, raw_customer):
    """
    Display prediction for a customer using both models
    Shows: customer info, probabilities, and final decision
    """
    print(f"\n📋 {title}")
    print("-" * 80)
    
    # Prepare customer data
    model_input = prepare_customer(raw_customer)
    
    # Get probabilities from both models
    lr_prob = log_reg.predict_proba(scaler.transform(model_input))[0, 1]
    rf_prob = random_forest.predict_proba(model_input)[0, 1]

    # Convert probabilities to Yes/No labels
    lr_label = "Yes" if lr_prob >= 0.5 else "No"
    rf_label = "Yes" if rf_prob >= 0.5 else "No"

    # Display results
    print(f"\n🤖 Model Predictions:")
    print(f"   Logistic Regression → Churn: {lr_label:3s} | Probability: {lr_prob:6.2%}")
    print(f"   Random Forest       → Churn: {rf_label:3s} | Probability: {rf_prob:6.2%}")

    # Final decision using selected model
    if final_model_name == "Logistic Regression":
        final_prob = lr_prob
        final_label = lr_label
    else:
        final_prob = rf_prob
        final_label = rf_label

    print(f"\n✅ Final Decision ({final_model_name}):")
    print(f"   Churn: {final_label} | Probability: {final_prob:.2%}")

def show_prediction_rf_final(title, raw_customer):
    """
    Display prediction using Random Forest only
    Used to showcase specific model behavior
    """
    print(f"\n📋 {title}")
    print("-" * 80)
    
    # Prepare customer data
    model_input = prepare_customer(raw_customer)
    
    # Get probabilities
    lr_prob = log_reg.predict_proba(scaler.transform(model_input))[0, 1]
    rf_prob = random_forest.predict_proba(model_input)[0, 1]

    lr_label = "Yes" if lr_prob >= 0.5 else "No"
    rf_label = "Yes" if rf_prob >= 0.5 else "No"

    print(f"\n🤖 Model Predictions:")
    print(f"   Logistic Regression → Churn: {lr_label:3s} | Probability: {lr_prob:6.2%}")
    print(f"   Random Forest       → Churn: {rf_label:3s} | Probability: {rf_prob:6.2%}")

    print(f"\n✅ Final Decision (Random Forest):")
    print(f"   Churn: {rf_label} | Probability: {rf_prob:.2%}")

# Example 1: Loyal, high-value customer (likely to STAY)
new_customer_no = {
    "CLIENTNUM": 999999999,
    "Attrition_Flag": "Existing Customer",
    "Customer_Age": 52,
    "Gender": "M",
    "Dependent_count": 2,
    "Education_Level": "Graduate",
    "Marital_Status": "Married",
    "Income_Category": "$80K - $120K",
    "Card_Category": "Blue",
    "Months_on_book": 56,
    "Total_Relationship_Count": 4,
    "Months_Inactive_12_mon": 1,
    "Contacts_Count_12_mon": 1,
    "Credit_Limit": 6000,
    "Total_Revolving_Bal": 2000,
    "Avg_Open_To_Buy": 4000,
    "Total_Amt_Chng_Q4_Q1": 0.92,
    "Total_Trans_Amt": 2744,
    "Total_Trans_Ct": 42,
    "Total_Ct_Chng_Q4_Q1": 0.85,
    "Avg_Utilization_Ratio": 0.33,
}

# Example 2: New, low-income customer with issues (likely to LEAVE)
new_customer_yes = {
    "CLIENTNUM": 888888888,
    "Attrition_Flag": "Attrited Customer",
    "Customer_Age": 39,
    "Gender": "F",
    "Dependent_count": 1,
    "Education_Level": "College",
    "Marital_Status": "Single",
    "Income_Category": "Less than $40K",
    "Card_Category": "Blue",
    "Months_on_book": 14,
    "Total_Relationship_Count": 1,
    "Months_Inactive_12_mon": 4,
    "Contacts_Count_12_mon": 4,
    "Credit_Limit": 3000,
    "Total_Revolving_Bal": 500,
    "Avg_Open_To_Buy": 2500,
    "Total_Amt_Chng_Q4_Q1": 0.45,
    "Total_Trans_Amt": 1232,
    "Total_Trans_Ct": 18,
    "Total_Ct_Chng_Q4_Q1": 0.30,
    "Avg_Utilization_Ratio": 0.79,
}

# Example 3: Stable family customer (likely to STAY)
new_customer_no_2 = {
    "CLIENTNUM": 777777777,
    "Attrition_Flag": "Existing Customer",
    "Customer_Age": 45,
    "Gender": "F",
    "Dependent_count": 3,
    "Education_Level": "High School",
    "Marital_Status": "Married",
    "Income_Category": "$40K - $60K",
    "Card_Category": "Blue",
    "Months_on_book": 48,
    "Total_Relationship_Count": 5,
    "Months_Inactive_12_mon": 0,
    "Contacts_Count_12_mon": 1,
    "Credit_Limit": 7500,
    "Total_Revolving_Bal": 3000,
    "Avg_Open_To_Buy": 4500,
    "Total_Amt_Chng_Q4_Q1": 1.10,
    "Total_Trans_Amt": 3600,
    "Total_Trans_Ct": 58,
    "Total_Ct_Chng_Q4_Q1": 0.95,
    "Avg_Utilization_Ratio": 0.28,
}

# Example 4: Very loyal, high-value customer (Random Forest decision)
new_customer_rf_final = {
    "CLIENTNUM": 666666666,
    "Attrition_Flag": "Existing Customer",
    "Customer_Age": 58,
    "Gender": "M",
    "Dependent_count": 1,
    "Education_Level": "Graduate",
    "Marital_Status": "Married",
    "Income_Category": "$80K - $120K",
    "Card_Category": "Blue",
    "Months_on_book": 72,
    "Total_Relationship_Count": 6,
    "Months_Inactive_12_mon": 0,
    "Contacts_Count_12_mon": 0,
    "Credit_Limit": 12000,
    "Total_Revolving_Bal": 5000,
    "Avg_Open_To_Buy": 7000,
    "Total_Amt_Chng_Q4_Q1": 1.25,
    "Total_Trans_Amt": 5200,
    "Total_Trans_Ct": 78,
    "Total_Ct_Chng_Q4_Q1": 1.10,
    "Avg_Utilization_Ratio": 0.22,
}

# Run predictions
show_prediction("Example 1: Loyal, High-Value Customer", new_customer_no)
show_prediction("Example 2: New Customer with Issues", new_customer_yes)
show_prediction("Example 3: Stable Family Customer", new_customer_no_2)
show_prediction_rf_final("Example 4: Very Loyal Customer (RF Decision)", new_customer_rf_final)

print("\n" + "="*80)
print("✅ PREDICTIONS COMPLETE")
print("="*80)


RAW CUSTOMER EXAMPLES

Example 1: No churn test
--------------------------------------------------------------------------------
New customer input:
   CLIENTNUM     Attrition_Flag  Customer_Age Gender  Dependent_count  \
0  999999999  Existing Customer            52      M                2   

  Education_Level Marital_Status Income_Category Card_Category  \
0        Graduate        Married    $80K - $120K          Blue   

   Months_on_book  ...  Months_Inactive_12_mon  Contacts_Count_12_mon  \
0              56  ...                       1                      1   

   Credit_Limit  Total_Revolving_Bal  Avg_Open_To_Buy  Total_Amt_Chng_Q4_Q1  \
0          6000                 2000             4000                  0.92   

   Total_Trans_Amt  Total_Trans_Ct  Total_Ct_Chng_Q4_Q1  Avg_Utilization_Ratio  
0             2744              42                 0.85                   0.33  

[1 rows x 21 columns]

Prediction output:
Logistic Regression -> Churn: No, Probability: 33.81%
Random